# HAM10000 Dataset Exploration

Before writing any model code, we explore the dataset to understand its structure,
class distribution, and any data quality issues.

## 1. Load Metadata

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

df = pd.read_csv('../data/HAM10000_metadata.csv')
print(f'Total samples: {len(df)}')
df.head()

## 2. Class Distribution

In [ ]:
class_counts = df['dx'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
class_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Sample count per diagnosis class')
ax.set_xlabel('Diagnosis')
ax.set_ylabel('Number of images')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print('Samples per class:')
print(class_counts.to_string())
print(f'\nImbalance ratio (most common / least common): {imbalance_ratio:.1f}x')

## 3. Sample Images per Class

In [ ]:
IMAGE_DIRS = [
    '../data/HAM10000_images_part_1',
    '../data/HAM10000_images_part_2',
]

def find_image(image_id):
    for d in IMAGE_DIRS:
        path = os.path.join(d, image_id + '.jpg')
        if os.path.exists(path):
            return path
    return None

classes = df['dx'].unique()
n_samples = 3

fig, axes = plt.subplots(len(classes), n_samples, figsize=(n_samples * 3, len(classes) * 3))

for row_idx, cls in enumerate(sorted(classes)):
    samples = df[df['dx'] == cls].sample(n=n_samples, random_state=42)
    for col_idx, (_, row) in enumerate(samples.iterrows()):
        path = find_image(row['image_id'])
        ax = axes[row_idx][col_idx]
        if path:
            ax.imshow(mpimg.imread(path))
        ax.axis('off')
        if col_idx == 0:
            ax.set_title(cls, fontsize=10, fontweight='bold')

plt.suptitle('Sample images per class (3 each)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Image Size Check

In [ ]:
from PIL import Image

sizes = set()
checked = 0

for image_id in df['image_id'].sample(n=200, random_state=0):
    path = find_image(image_id)
    if path:
        with Image.open(path) as img:
            sizes.add(img.size)
        checked += 1

print(f'Checked {checked} images')
print(f'Unique dimensions (width x height): {sizes}')

## 5. Missing Values

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing.to_string())

if missing.sum() == 0:
    print('\nNo missing values found.')
else:
    print(f'\nTotal missing cells: {missing.sum()}')

## 6. Key Findings

**Total images:** 10,015  
**Number of classes:** 7

**Class breakdown:**
| Code | Name | Description |
|------|------|-------------|
| nv | Melanocytic nevus | Benign mole — majority class (~67%) |
| mel | Melanoma | Malignant skin cancer |
| bkl | Benign keratosis | Non-cancerous growth |
| bcc | Basal cell carcinoma | Common skin cancer |
| akiec | Actinic keratosis | Pre-cancerous lesion |
| vasc | Vascular lesion | Blood vessel lesion |
| df | Dermatofibroma | Benign skin tumour |

**Why class imbalance is a problem:**  
A model trained naively on this data will learn to predict `nv` (moles) for almost everything,
because doing so gets it ~67% accuracy without learning anything useful. The dangerous
failure mode is missing `mel` (melanoma) — a false negative in a real diagnostic tool
has serious consequences. We fix this in training using `WeightedRandomSampler`, which
oversamples the minority classes so each training batch is more balanced.